# Changing the Default Comparator per Type

`annotate_xeval` assigns a default comparator to every leaf by its JSON type: `exact` for
strings and booleans, `numeric` for numbers and integers. This notebook shows how to change
that default for a whole type with `set_type_default`, instead of editing fields one by one.


## Change Global Type Defaults

By default, `annotate_xeval` assigns `exact` to every string field and `numeric` to every number field. If you want a different default for an entire type — applied to every matching field without touching them one by one — use `set_type_default` before calling `annotate_xeval`.

Per-field `x-eval-compare` still wins over the type default, so this is a "set the baseline, override where needed" pattern. Useful when you have many fields of one type and the built-in default isn't right for your domain.

**Caveat:** `set_type_default` mutates process-wide state. Always pair it with `reset_type_defaults()` so it doesn't leak into other evaluations in the same session.

**Example:** make every number field default to a comparator that rounds to int before comparing (useful when your domain treats numeric values as whole units).

In [1]:
# Minimal self-contained setup
GOLD = [
    {"material": "Silicon", "temperature": 300.0, "thickness": 10.5},
    {"material": "Gold",    "temperature": 200.0, "thickness": 8.0},
]
EXTRACTED = [
    {"material": "Silicon", "temperature": 300.4,  "thickness": 10.2},
    {"material": "Gold",    "temperature": 200.0, "thickness": 8.0},
]

import json

from json_extract_eval import (
    infer_schema,
    annotate_xeval,
    evaluate,
    set_type_default,
    reset_type_defaults,
)
from json_extract_eval.core.comparators.comparator import ComparatorResult
from json_extract_eval.core.comparators.registry import register

def compare_numeric_int(gold, extracted, params):
    """Round both values to the nearest int, then compare."""
    try:
        g, e = round(float(gold)), round(float(extracted))
        return ComparatorResult(score=1.0 if g == e else 0.0, comparator="numeric_int")
    except (ValueError, TypeError):
        return ComparatorResult(score=0.0, comparator="numeric_int")


register("numeric_int", compare_numeric_int, overwrite=True)

# Make every number field default to numeric_int
set_type_default("number", "numeric_int")

schema = infer_schema(GOLD)
annotate_xeval(schema)

print(json.dumps(schema, indent=2))

# Reset so this doesn't leak into other evaluations
reset_type_defaults()

{
  "type": "object",
  "properties": {
    "material": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "temperature": {
      "type": "number",
      "x-eval-compare": "numeric_int"
    },
    "thickness": {
      "type": "number",
      "x-eval-compare": "numeric_int"
    }
  }
}


The eval schema now has `numeric_int` as the default comparator for all number fields (`temperature` and `thickness`), which originally would have been `numeric`. The `material` field is a string, so it still gets `exact`.